# Module 6 project: phase estimation of 1/3 with noise

**Quantum Computing Intermediate · Module 6 · about 180 minutes** · notebook version 2026-10-05

**The question of this experiment.** Phase estimation reads a phase as an m-bit binary fraction, so more counting qubits give a finer answer. But the circuit grows with m: m controlled phases, an inverse QFT with m(m − 1)/2 more, and m readouts. **On a noisy computer, how many counting qubits give the most accurate estimate of 1/3, and how does the answer depend on the noise?**

In this project you run an experiment on a **noisy simulator**: a simulator that adds the errors of a real quantum computer, described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain each result is, and say what the experiment cannot tell you. These are the skills of course outcome 5.

**The plan.**

1. Write three statistics functions (Step 1) and meet the course noise model (Step 2).
2. Write `qpe_circuit(m)` and `estimate_error(counts, m)` (Step 3).
3. Run it once on the noisy simulator and compare with the noise model's expected values (Step 4).
4. Measure the accuracy for 3 to 7 counting qubits, with uncertainties (Step 5), and test which differences are significant (Step 6).
5. Find the best number of counting qubits for different noise levels (Step 7).
6. Run your own computer's noise model and get your verification value (Step 8), then look at the limits.

**How it is graded.** Your project quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell, three results from this notebook, and four questions about uncertainty, interpretation and limits. Your project quiz gives you three numbers: COUNT (the number of counting qubits, 3 to 6), and P2 and READOUT for your computer.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import (QuantumCircuit, Statevector, AerSimulator, NoiseModel, depolarizing_error, ReadoutError,
                  simulate_density_matrix)

print("Ready. qsim", qsim.__version__, "and NumPy", np.__version__)

## Step 1: how certain is a result from shots?

A circuit that gives a result with probability p, run for S shots, gives that result k times. The number k is random: it follows the **binomial distribution**. The fraction f = k / S has

- mean p, and
- standard deviation **σ = √(p (1 − p) / S)**. This spread is called **shot noise**.

For example, with S = 4,000 shots and p = 0.559, σ = 0.0079: two runs of the same circuit typically differ by about this much. Four times as many shots halve σ.

**The course rule: 3σ.** A measured fraction f agrees with an expected value p when |f − p| ≤ 3σ. For a noisy simulator, p must be the **noise model's** expected value (computed exactly, without shots, in Step 2), not the ideal value from a circuit without noise. A correct run lands outside 3σ only about 3 times in 1,000 (0.27%).

**Comparing two results.** Two fractions f₁ and f₂, each from S independent shots, differ by f₂ − f₁, with standard deviation

**σ_diff = √( f₁(1 − f₁)/S + f₂(1 − f₂)/S )**.

Uncertainties of independent results add **in quadrature** (their squares add), so a difference is less certain than either result alone. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but this experiment cannot show it.

**Your task:** write the three functions below. They are used in every later step, and the check cell tests them.

- `shot_sigma(p, shots)`: σ = √(p (1 − p) / shots), the standard deviation of a fraction.
- `within_3_sigma(k, shots, p)`: `True` if the fraction k / shots is within 3 · shot_sigma(p, shots) of p (in either direction), otherwise `False`.
- `difference_sigma(f1, f2, shots)`: σ_diff for two fractions from `shots` shots each.

In [ ]:
def shot_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete shot_sigma() first.")


def within_3_sigma(k, shots, p):
    """True if k successes out of `shots` agree with the probability p by the course's 3-sigma rule."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete within_3_sigma() first.")


def difference_sigma(f1, f2, shots):
    """The standard deviation of f2 - f1, for two fractions from `shots` independent shots each."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete difference_sigma() first.")

In [ ]:
print("shot_sigma(0.687, 4000) =", round(shot_sigma(0.687, 4000), 5), "  (expected 0.00733)")
print("within_3_sigma(2748, 4000, 0.687) =", within_3_sigma(2748, 4000, 0.687), "  (expected True: exactly 0.687)")
print("within_3_sigma(2655, 4000, 0.687) =", within_3_sigma(2655, 4000, 0.687), "  (expected False: 3.2 sigma too low)")
print("difference_sigma(0.687, 0.634, 4000) =", round(difference_sigma(0.687, 0.634, 4000), 5), "  (expected 0.01057)")

## Step 2: the course noise model

A **noise model** says what can go wrong in each operation. The course model has three parts, each with a number:

- after every **two-qubit gate** (`cx`, `cz`, `cp`, `rzz`, `swap`), a **depolarizing error** with parameter P2: with probability P2 the two qubits are replaced by a completely random state. For a density matrix, ρ → (1 − P2) ρ + P2 · I/4 on those two qubits. (An average gate error, the number IBM reports, of 0.75 · P2.)
- after every **one-qubit gate**, the same on one qubit with parameter P2 / 10: one-qubit gates are about ten times better.
- at **readout**: a qubit in |0⟩ is recorded as 1 with probability READOUT, and a qubit in |1⟩ is recorded as 0 with probability 2 · READOUT. The second is larger because an excited qubit can decay to |0⟩ during the measurement.

The course values are **P2 = 0.01** and **READOUT = 0.02**. They are several times worse than IBM's best current computers (the model of `ibm_pittsburgh` used in Module 4 has a median two-qubit gate error of 0.0015 and a median readout error of 0.0045), so that the effects of noise show clearly in a few thousand shots. In Canvas, your project quiz gives you your own P2 (0.005 to 0.030) and READOUT (0.005 to 0.030): a model of "your" computer.

The noise model applies errors by **gate name**. A gate on three qubits, such as `ccx` or `mcx`, would get no error at all, so every circuit in this project uses only one- and two-qubit gates.

The prepared cell builds the noise model with the same functions as Qiskit Aer (`qiskit_aer.noise`), and defines `expected_probabilities(qc, p2, readout)`. It gives the noise model's **exact** probability of every recorded result, with no shots and no randomness: it follows the **density matrix**, the mixed state that the noisy gates produce, then applies the readout errors to the recorded bits. A run with shots, `AerSimulator(noise_model=...).run(qc, shots=4000)`, samples from these same probabilities and adds shot noise. The 3σ rule compares the two.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]


def course_noise_model(p2=0.01, readout=0.02):
    """Prepared: depolarizing error p2 after two-qubit gates and p2/10 after one-qubit gates; readout errors."""
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    nm.add_all_qubit_readout_error(ReadoutError([[1 - readout, readout], [2 * readout, 1 - 2 * readout]]))
    return nm


def expected_probabilities(qc, p2=0.01, readout=0.02):
    """Prepared: the noise model's exact probability of each recorded result, as a dict {"bits": probability}.

    qc must end with its measurements (classical bit j receives one qubit). No shots are taken.
    """
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    rho = simulate_density_matrix(body, course_noise_model(p2, readout))   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])      # index bit j = classical bit j
    confusion = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])   # rows: true bit; columns: recorded bit
    for j in range(m):                                                     # each recorded bit can be flipped
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, confusion).reshape(-1)
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


print(course_noise_model())

## Step 3: the phase estimation circuit

The phase gate P(2π/3) has the eigenvector |1⟩ with eigenvalue e^(2πi·θ), θ = 1/3. Phase estimation with m counting qubits (qubits 0 to m − 1) and the target (qubit m) works as in Module 1:

1. X on the target, so that it holds the eigenvector |1⟩;
2. H on every counting qubit;
3. for each counting qubit k, a controlled phase `cp(2 * pi * theta * 2**k, k, m)`: the phase gate applied 2ᵏ times, written as one gate with 2ᵏ times the angle;
4. the inverse QFT on the counting qubits (prepared, as in Module 1, with `h`, `cp` and `swap`);
5. measure counting qubit k into classical bit k. The target is not measured.

The result y, read as a binary number, gives the estimate θ ≈ y / 2ᵐ.

**Your task (a):** write `qpe_circuit(m)` for θ = 1/3. Use one `cp` per counting qubit: repeating the gate 2ᵏ times gives the same ideal result but many more gates, and so more noise.

**Your task (b):** write `estimate_error(counts, m)`: the **mean error** of the estimates over all shots, Σ count · error(y) / shots, where error(y) is the distance between y / 2ᵐ and 1/3 **around the circle**: a phase is only defined up to a whole turn, so with d = |y / 2ᵐ − 1/3|, the error is min(d, 1 − d). Read y with `int(key, 2)`.

In [ ]:
THETA = 1 / 3


def inverse_qft(qc, qubits):
    """Prepared: the inverse quantum Fourier transform on the given qubits (with the swaps), as in Module 1."""
    qubits = list(qubits)
    n = len(qubits)
    for j in range(n // 2):
        qc.swap(qubits[j], qubits[n - 1 - j])
    for j in range(n):
        for k in range(j):
            qc.cp(-np.pi / 2 ** (j - k), qubits[k], qubits[j])
        qc.h(qubits[j])


def qpe_circuit(m):
    """Phase estimation of THETA with m counting qubits (0 to m - 1) and the target qubit m."""
    qc = QuantumCircuit(m + 1, m)
    # YOUR CODE HERE
    raise NotImplementedError("Complete qpe_circuit() first.")
    return qc


def estimate_error(counts, m):
    """The mean distance, around the circle, between y / 2**m and THETA over all shots."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete estimate_error() first.")

In [ ]:
def as_counts(probs, shots=10 ** 6):
    """Prepared: probabilities scaled to look like counts, so that estimate_error() can use them too."""
    return {k: v * shots for k, v in probs.items()}


def best_estimate(m):
    """Prepared: the m-bit result y whose y / 2**m is closest to THETA."""
    return min(range(2 ** m), key=lambda y: min(abs(y / 2 ** m - THETA), 1 - abs(y / 2 ** m - THETA)))


qc = qpe_circuit(4)
ideal = expected_probabilities(qc, 0, 0)
best = format(best_estimate(4), "04b")
print("ideal probabilities with 4 counting qubits (largest first):")
for y, p in sorted(ideal.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {y} = {int(y, 2):2d}/16 = {int(y, 2) / 16:.4f}   probability {p:.4f}")
print("best estimate:", best, "=", int(best, 2) / 16, "  ideal P(best) =", round(ideal[best], 4), "  (expected 0.6849)")
print("ideal mean error:", round(estimate_error(as_counts(ideal), 4), 4), "  (expected 0.0461)")
print("gates:", qc.count_ops())

**What to notice.** 1/3 = 0.010101... in binary never ends, so no m-bit fraction equals it. With 4 counting qubits the closest is 0101 = 5/16 = 0.3125, which comes out with probability 0.685; the rest spreads over nearby values (6/16 with 0.17, 4/16 with 0.04, 7/16 with 0.03, ...). This is not noise: it is how phase estimation behaves for a phase that is not an m-bit fraction, and the ideal probability of the best estimate stays near 0.68 for every m. The mean error still falls as m grows, because the estimates get finer.

## Step 4: one run on the noisy simulator

With the course noise model (P2 = 0.01, READOUT = 0.02) and 4 counting qubits, the cell computes the noise model's expected probability of the best estimate 0101 and its expected mean error, runs 4,000 shots, and compares.

The mean error is an average over shots of a quantity that varies from shot to shot, so its uncertainty is the **standard error of the mean**: SEM = s / √shots, where s is the standard deviation of the error over the shots. The prepared `error_sem(counts, m)` computes it.

`RUN_SEED` fixes the simulator's random numbers. The project quiz asks for your measured fraction of 0101 (any run with 4,000 shots) and for the expected mean error.

In [ ]:
def error_sem(counts, m):
    """Prepared: the standard error of the mean error: (standard deviation of the per-shot error) / sqrt(shots)."""
    errs = np.array([min(abs(int(y, 2) / 2 ** m - THETA), 1 - abs(int(y, 2) / 2 ** m - THETA)) for y in counts])
    n = np.array([counts[y] for y in counts], dtype=float)
    mean = np.sum(n * errs) / n.sum()
    return float(np.sqrt(np.sum(n * (errs - mean) ** 2) / (n.sum() - 1) / n.sum()))


RUN_SEED = 1                     # any whole number; change it to see another run
sim = AerSimulator(noise_model=course_noise_model(0.01, 0.02))

qc = qpe_circuit(4)
expected = expected_probabilities(qc, 0.01, 0.02)
p_expected = expected[best]
err_expected = estimate_error(as_counts(expected), 4)
counts = sim.run(qc, shots=4000, seed_simulator=RUN_SEED).result().get_counts()
k = counts.get(best, 0)
print(f"P(best = {best}): ideal {ideal[best]:.4f}, noise model expected {p_expected:.4f}, measured {k / 4000:.4f}")
print(f"  sigma = {shot_sigma(p_expected, 4000):.4f}; within 3 sigma: {within_3_sigma(k, 4000, p_expected)}")
print(f"mean error: ideal 0.0461, noise model expected {err_expected:.4f}, measured {estimate_error(counts, 4):.4f} ± {error_sem(counts, 4):.4f} (SEM)")

labels = [format(i, "04b") for i in range(16)]
x = np.arange(16)
plt.figure(figsize=(8, 3.2))
plt.bar(x - 0.2, [expected.get(s, 0) for s in labels], 0.4, label="noise model, expected")
plt.bar(x + 0.2, [counts.get(s, 0) / 4000 for s in labels], 0.4, label="4000 shots")
plt.xticks(x, labels, rotation=90); plt.ylabel("fraction of shots"); plt.legend(); plt.tight_layout(); plt.show()

**What to notice.** The noise lowers P(0101) from 0.685 to 0.559 and raises the mean error from 0.046 to 0.079: the probability it takes away spreads over all 16 results, including ones far from 1/3. A completely random result would have a mean error of 0.25 (the average distance around the circle). Your measured fraction should be within about 0.024 (3σ) of 0.559.

## Step 5: accuracy against the number of counting qubits

The cell builds the circuit for m = 3 to 7 counting qubits and, for each, computes the ideal and noise-model mean errors and measures 4,000 shots. The plot shows the measured mean error with error bars of ±3 SEM. It takes up to a minute: with 7 counting qubits the simulator follows an 8-qubit density matrix.

In [ ]:
ms = [3, 4, 5, 6, 7]
rows = []
for m in ms:
    qc = qpe_circuit(m)
    ideal_err = estimate_error(as_counts(expected_probabilities(qc, 0, 0)), m)
    exp_err = estimate_error(as_counts(expected_probabilities(qc, 0.01, 0.02)), m)
    cts = sim.run(qc, shots=4000, seed_simulator=RUN_SEED + 10 + m).result().get_counts()
    two_q = sum(v for g, v in qc.count_ops().items() if g in TWO_QUBIT_GATES)
    rows.append((m, two_q, ideal_err, exp_err, estimate_error(cts, m), error_sem(cts, m)))

print(" m   two-qubit gates   ideal error   expected (noise)   measured ± SEM")
for m, two_q, ie, ee, me, se in rows:
    print(f" {m}        {two_q:2d}           {ie:.4f}         {ee:.4f}          {me:.4f} ± {se:.4f}")
measured_err = {m: (me, se) for m, _, _, _, me, se in rows}

plt.figure(figsize=(7, 3.5))
plt.plot(ms, [r[2] for r in rows], "k--o", label="ideal")
plt.plot(ms, [r[3] for r in rows], "C0-s", label="noise model, expected")
plt.errorbar(ms, [r[4] for r in rows], yerr=[3 * r[5] for r in rows], fmt="C3o", capsize=4, label="4000 shots, ±3 SEM")
plt.xlabel("counting qubits m"); plt.ylabel("mean error of the estimate"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

**What to notice.** Without noise the mean error roughly halves with each counting qubit (0.079, 0.046, 0.026, 0.015, 0.008). With noise it falls much more slowly (0.101, 0.079, 0.067, 0.063, 0.062): the finer resolution is paid for with more gates (7 two-qubit gates for m = 3, 31 for m = 7) and more readouts, and their errors add a background of random results that does not shrink. Under the course noise model, going from 6 to 7 counting qubits gains almost nothing.

## Step 6: which differences are significant?

For two mean errors with standard errors SEM₁ and SEM₂, the difference has σ_diff = √(SEM₁² + SEM₂²): the same rule as `difference_sigma`, which does this for fractions. The cell tests two pairs from Step 5: m = 4 against 5, and m = 6 against 7.

In [ ]:
for a, b in ((4, 5), (6, 7)):
    (ea, sa), (eb, sb) = measured_err[a], measured_err[b]
    d, sd = eb - ea, math.sqrt(sa ** 2 + sb ** 2)
    print(f"m = {a}: {ea:.4f}   m = {b}: {eb:.4f}   difference {d:+.4f}, sigma_diff {sd:.4f} = {d / sd:+.1f} sigma_diff,",
          "significant" if abs(d) > 3 * sd else "not significant")

**What to notice.** Going from 4 to 5 counting qubits lowers the expected mean error by 0.013, about 5 σ_diff: significant. From 6 to 7 the expected gain is 0.0008, well below σ_diff (about 0.003): 4,000 shots cannot say whether 7 counting qubits are better than 6 under this noise, and with a different RUN_SEED the measured order of the two can flip.

## Step 7: the best number of counting qubits depends on the noise

The cell computes the noise model's expected mean error for m = 3 to 7 at six noise levels (READOUT = 0.02 throughout; no shots) and prints the best m for each. The project quiz asks for the best m at P2 = 0.020. It takes up to a minute.

In [ ]:
p2_levels = [0.005, 0.010, 0.015, 0.020, 0.025, 0.030]
best_m = {}
print("P2      " + "   ".join(f"m={m}  " for m in ms) + "  best m")
for p in p2_levels:
    errs = {m: estimate_error(as_counts(expected_probabilities(qpe_circuit(m), p, 0.02)), m) for m in ms}
    best_m[p] = min(errs, key=errs.get)
    print(f"{p:.3f}   " + "   ".join(f"{errs[m]:.4f}" for m in ms) + f"    {best_m[p]}")

**What to notice.** At P2 = 0.005 the error still falls up to m = 7; at P2 = 0.015 the best is 6, and from P2 = 0.020 on it is 5. The noisier the computer, the fewer counting qubits are worth using: past the best m, each extra qubit adds more noise than precision. The best m also depends on READOUT, since every counting qubit is read out.

## Step 8: your computer, and your verification value

Open your **project quiz for this experiment** in Canvas. Question 1 shows your COUNT (3 to 6 counting qubits), your P2 and your READOUT. Type them below and run the cells.

The first cell runs your circuit on **your** noise model with 4,000 shots and keeps the counts as `my_counts`. The check cell tests your three statistics functions, `qpe_circuit()` and `estimate_error()`, and checks that `my_counts` agree with your noise model by the 3σ rule. Only if everything passes does it print your **verification value**: 10,000 times the noise model's exact probability of the best estimate with COUNT counting qubits on your computer, rounded to a whole number.

In [ ]:
COUNT = 0        # your number from Canvas, for example 5
P2 = 0.0         # your number from Canvas, for example 0.017
READOUT = 0.0    # your number from Canvas, for example 0.015
my_counts = {}   # filled by the next cell

In [ ]:
qc = qpe_circuit(int(COUNT))
my_expected = expected_probabilities(qc, P2, READOUT)
my_best = format(best_estimate(int(COUNT)), f"0{int(COUNT)}b")
my_counts = AerSimulator(noise_model=course_noise_model(P2, READOUT)).run(qc, shots=4000, seed_simulator=RUN_SEED + 100).result().get_counts()
kk = my_counts.get(my_best, 0)
print(f"best estimate {my_best}: expected {my_expected[my_best]:.4f}, measured {kk / 4000:.4f} ± {shot_sigma(my_expected[my_best], 4000):.4f} (1 sigma)")
print(f"mean error: expected {estimate_error(as_counts(my_expected), int(COUNT)):.4f}, "
      f"measured {estimate_error(my_counts, int(COUNT)):.4f} ± {error_sem(my_counts, int(COUNT)):.4f} (SEM)")

In [ ]:
# The course's check code. You do not need to read it; it is here so that you can.
import math

import numpy as np

try:                                    # real Qiskit, if it is installed
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
except ImportError:                     # the browser: the course simulator
    from qsim import QuantumCircuit, Statevector

ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
SHOTS = 4000

# ---------------------------------------------------------------- an exact noisy simulator (NumPy only)
_S2 = 1 / math.sqrt(2)
_FIXED = {
    "h": np.array([[_S2, _S2], [_S2, -_S2]], dtype=complex),
    "x": np.array([[0, 1], [1, 0]], dtype=complex),
    "t": np.diag([1, np.exp(1j * math.pi / 4)]),
    "tdg": np.diag([1, np.exp(-1j * math.pi / 4)]),
    "s": np.diag([1, 1j]),
    "sdg": np.diag([1, -1j]),
}


def _matrix(name, params):
    """The gate's matrix in Qiskit's convention (for two-qubit gates, the first qubit is the right-hand factor)."""
    if name in _FIXED:
        return _FIXED[name]
    if name == "rx":
        c, s = math.cos(params[0] / 2), math.sin(params[0] / 2)
        return np.array([[c, -1j * s], [-1j * s, c]])
    if name == "rz":
        return np.diag([np.exp(-0.5j * params[0]), np.exp(0.5j * params[0])])
    if name == "p":
        return np.diag([1, np.exp(1j * params[0])])
    if name == "cx":                     # control = first qubit (the low bit of the 4 x 4 index)
        return np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]], dtype=complex)
    if name == "cz":
        return np.diag([1, 1, 1, -1]).astype(complex)
    if name == "cp":
        return np.diag([1, 1, 1, np.exp(1j * params[0])])
    if name == "rzz":
        a = np.exp(-0.5j * params[0])
        b = np.exp(0.5j * params[0])
        return np.diag([a, b, b, a])
    if name == "swap":
        return np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)
    raise ValueError(f"unknown gate {name}")


def _apply(rho, u, qubits, n):
    """rho -> U rho U^dagger, with U acting on the given qubits (qubits[0] is the low bit of U's index)."""
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    # axis of qubit q in the row index is n - 1 - q; in the column index 2n - 1 - q
    ut = u.reshape([2] * (2 * k))           # axes: out bits (high to low), in bits (high to low)
    row_axes = [n - 1 - q for q in reversed(qubits)]
    col_axes = [2 * n - 1 - q for q in reversed(qubits)]
    t = np.tensordot(ut, t, axes=(list(range(k, 2 * k)), row_axes))
    t = np.moveaxis(t, list(range(k)), row_axes)
    t = np.tensordot(t, ut.conj(), axes=(col_axes, list(range(k, 2 * k))))
    t = np.moveaxis(t, list(range(2 * n - k, 2 * n)), col_axes)
    return t.reshape(2 ** n, 2 ** n)


def _depolarize(rho, lam, qubits, n):
    """rho -> (1 - lam) rho + lam (I / 2^k on the given qubits) x (rho traced over them)."""
    if lam == 0:
        return rho
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    row_axes = [n - 1 - q for q in qubits]
    col_axes = [2 * n - 1 - q for q in qubits]
    letters = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    sub = list(letters[:2 * n])
    for ra, ca in zip(row_axes, col_axes):
        sub[ca] = sub[ra]
    keep = [i for i in range(2 * n) if i not in row_axes + col_axes]
    red = np.einsum("".join(sub) + "->" + "".join(sub[i] for i in keep), t)
    eye = np.eye(2 ** k).reshape([2] * (2 * k))      # identity: row bit i pairs with column bit i
    full = np.multiply.outer(red, eye)
    full = np.moveaxis(full, list(range(len(keep), 2 * n)), row_axes[::-1] + col_axes[::-1])
    return (1 - lam) * rho + lam * full.reshape(2 ** n, 2 ** n) / 2 ** k


def exact_probabilities(ops, n, measured, p2, readout):
    """Exact probabilities of the recorded results under the course noise model.

    ops: list of (name, qubits, params) applied to |0...0>; measured: the qubits measured at the end, measured[j] into
    classical bit j. Returns an array over the 2^len(measured) recorded results (bit j of the index is classical bit j).
    """
    rho = np.zeros((2 ** n, 2 ** n), dtype=complex)
    rho[0, 0] = 1
    for name, qubits, params in ops:
        rho = _apply(rho, _matrix(name, params), list(qubits), n)
        if name in TWO_QUBIT_GATES:
            rho = _depolarize(rho, p2, list(qubits), n)
        elif name in ONE_QUBIT_GATES:
            rho = _depolarize(rho, p2 / 10, list(qubits), n)
    probs = np.real(np.diag(rho)).clip(min=0)
    probs = probs / probs.sum()
    m = len(measured)
    idx = np.arange(2 ** n)
    code = np.zeros(2 ** n, dtype=int)
    for j, q in enumerate(measured):
        code |= ((idx >> q) & 1) << j
    dist = np.zeros(2 ** m)
    np.add.at(dist, code, probs)
    conf = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])
    t = dist.reshape([2] * m)
    for j in range(m):
        ax = m - 1 - j
        t = np.moveaxis(np.tensordot(t, conf, axes=([ax], [0])), -1, ax)
    dist = t.reshape(-1).clip(min=0)
    return dist / dist.sum()


def build_circuit(ops, n, measured):
    """The ops as a QuantumCircuit (Qiskit or qsim), with the measurements at the end."""
    qc = QuantumCircuit(n, len(measured)) if measured else QuantumCircuit(n)
    for name, qubits, params in ops:
        getattr(qc, name)(*params, *qubits)
    for j, q in enumerate(measured):
        qc.measure(q, j)
    return qc


# ---------------------------------------------------------------- helpers for the checks
def _op(inst):
    return getattr(inst, "operation", inst)


def _name(inst):
    return getattr(_op(inst), "name", "")


def _qubit_indices(qc, inst):
    qs = getattr(inst, "qubits", ())
    try:
        return [qc.find_bit(q).index for q in qs]
    except Exception:  # noqa: BLE001
        return [int(q) for q in qs]


def _clbit_indices(qc, inst):
    cs = getattr(inst, "clbits", ())
    try:
        return [qc.find_bit(c).index for c in cs]
    except Exception:  # noqa: BLE001
        return [int(c) for c in cs]


def measurement_map(qc):
    """{qubit: classical bit} for the measurements in a circuit."""
    return {_qubit_indices(qc, i)[0]: _clbit_indices(qc, i)[0] for i in qc.data if _name(i) == "measure"}


def big_gates(qc):
    """Names of gates on 3 or more qubits (the course noise model puts no error on them)."""
    return sorted({_name(i) for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) >= 3})


def two_qubit_count(qc):
    """The number of two-qubit gates in a circuit (barriers not counted)."""
    return sum(1 for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) == 2)


def ops_two_qubit_count(ops):
    return sum(1 for name, qubits, params in ops if len(qubits) == 2)


def ideal_probabilities(qc):
    """Probabilities of all qubits' values (index bit q = qubit q) for a circuit without its final measurements."""
    c = qc.copy()
    c.remove_final_measurements()
    return np.abs(np.asarray(Statevector(c).data, dtype=complex)) ** 2


def counts_total(counts):
    try:
        return int(sum(int(v) for v in counts.values()))
    except Exception:  # noqa: BLE001
        return -1


# ---------------------------------------------------------------- the statistics functions
def _close(a, b, tol=1e-9):
    try:
        return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
    except (TypeError, ValueError):
        return False


def test_statistics(shot_sigma, within_3_sigma, difference_sigma):
    """Returns (passed, messages)."""
    msgs = []
    cases = [(0.5, 100), (0.687, 4000), (0.1, 1000), (0.0, 50), (1.0, 50), (0.9206, 4000)]
    for p, n in cases:
        want = math.sqrt(p * (1 - p) / n)
        try:
            got = shot_sigma(p, n)
        except NotImplementedError:
            return False, ["shot_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"shot_sigma({p}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            hint = ""
            if _close(got, math.sqrt(p * (1 - p) * n)):
                hint = " That is the uncertainty of the count k; divide by shots for the fraction: sqrt(p * (1 - p) / shots)."
            elif _close(got, p * (1 - p) / n):
                hint = " Take the square root: sqrt(p * (1 - p) / shots)."
            elif _close(got, math.sqrt(p / n)) or _close(got, math.sqrt(1 / n)):
                hint = " Use p * (1 - p), the binomial variance of one shot."
            return False, [f"shot_sigma({p}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("shot_sigma(): passed (6 cases).")

    w_cases = [(2748, 4000, 0.687, True), (2655, 4000, 0.687, False), (2841, 4000, 0.687, False),
               (2700, 4000, 0.687, True), (2821, 4000, 0.687, True), (55, 100, 0.5, True), (66, 100, 0.5, False),
               (34, 100, 0.5, False), (3700, 4000, 0.9206, True), (3620, 4000, 0.9206, False)]
    for k, n, p, want in w_cases:
        try:
            got = within_3_sigma(k, n, p)
        except NotImplementedError:
            return False, msgs + ["within_3_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) raised {type(e).__name__}: {e}"]
        if bool(got) != want or not isinstance(got, (bool, np.bool_)):
            d = abs(k / n - p) / max(math.sqrt(p * (1 - p) / n), 1e-300)
            hint = ""
            if not isinstance(got, (bool, np.bool_)):
                hint = " Return True or False."
            elif k > n * p and not want:
                hint = " Check both directions: a result can be too high as well as too low (use abs())."
            elif 2 < d <= 3:
                hint = " The course rule allows up to 3 standard deviations."
            else:
                hint = " Compare the fraction k / shots with p, using shot_sigma(p, shots)."
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) returned {got}; expected {want} "
                                  f"(the result is {d:.2f} standard deviations from p).{hint}"]
    msgs.append("within_3_sigma(): passed (10 cases).")

    d_cases = [(0.687, 0.634, 4000), (0.5, 0.5, 100), (0.9, 0.1, 1000), (0.2, 0.25, 4000)]
    for f1, f2, n in d_cases:
        want = math.sqrt(f1 * (1 - f1) / n + f2 * (1 - f2) / n)
        try:
            got = difference_sigma(f1, f2, n)
        except NotImplementedError:
            return False, msgs + ["difference_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            s1, s2 = math.sqrt(f1 * (1 - f1) / n), math.sqrt(f2 * (1 - f2) / n)
            hint = ""
            if _close(got, s1 + s2):
                hint = " Uncertainties add in quadrature: sqrt(sigma1**2 + sigma2**2), not sigma1 + sigma2."
            elif _close(got, abs(s1 - s2)):
                hint = " The uncertainty of a difference is larger than either one: sqrt(sigma1**2 + sigma2**2)."
            elif _close(got, want ** 2):
                hint = " Take the square root."
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("difference_sigma(): passed (4 cases).")
    return True, msgs


def personal_noise_ok(P2, READOUT):
    """Checks the personal noise numbers from Canvas; returns (ok, message, p2, readout)."""
    try:
        p2, ro = float(P2), float(READOUT)
    except (TypeError, ValueError):
        return False, "P2 and READOUT must be the numbers shown in your Canvas project quiz.", None, None
    if not (0.005 - 1e-12 <= p2 <= 0.030 + 1e-12) or abs(round(p2, 3) - p2) > 1e-9:
        return False, "P2 is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    if not (0.005 - 1e-12 <= ro <= 0.030 + 1e-12) or abs(round(ro, 3) - ro) > 1e-9:
        return False, "READOUT is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    return True, "", round(p2, 3), round(ro, 3)


PHASE = 1 / 3


def reference_iqft_ops(qubits):
    qubits = list(qubits)
    n = len(qubits)
    ops = [("swap", [qubits[j], qubits[n - 1 - j]], []) for j in range(n // 2)]
    for j in range(n):
        for k in range(j):
            ops.append(("cp", [qubits[k], qubits[j]], [-math.pi / 2 ** (j - k)]))
        ops.append(("h", [qubits[j]], []))
    return ops


def reference_qpe_ops(m, phase=PHASE):
    ops = [("x", [m], [])] + [("h", [k], []) for k in range(m)]
    ops += [("cp", [k, m], [2 * math.pi * phase * 2 ** k]) for k in range(m)]
    return ops + reference_iqft_ops(range(m))


def reference_qpe(m, phase=PHASE):
    return build_circuit(reference_qpe_ops(m, phase), m + 1, list(range(m)))


def circular_distance(y, m, phase=PHASE):
    d = abs(y / 2 ** m - phase) % 1
    return min(d, 1 - d)


def best_estimate(m, phase=PHASE):
    return min(range(2 ** m), key=lambda y: circular_distance(y, m, phase))


def noisy_distribution(m, p2, readout):
    return exact_probabilities(reference_qpe_ops(m), m + 1, list(range(m)), p2, readout)


def expected_best(m, p2, readout):
    return float(noisy_distribution(m, p2, readout)[best_estimate(m)])


def expected_error(m, p2, readout):
    d = noisy_distribution(m, p2, readout)
    return float(sum(d[y] * circular_distance(y, m) for y in range(2 ** m)))


def _counting_probs(qc, m):
    probs = ideal_probabilities(qc)
    out = np.zeros(2 ** m)
    for idx, pr in enumerate(probs):
        out[idx % 2 ** m] += pr
    return out


def _test_qpe(qpe_circuit):
    for m in (2, 3, 4, 5):
        try:
            qc = qpe_circuit(m)
        except NotImplementedError:
            return False, ["qpe_circuit() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"qpe_circuit({m}) raised {type(e).__name__}: {e}"]
        if not hasattr(qc, "data") or getattr(qc, "num_qubits", None) != m + 1:
            return False, [f"qpe_circuit({m}) should return a QuantumCircuit on {m + 1} qubits ({m} counting qubits and the target)."]
        big = big_gates(qc)
        if big:
            return False, [f"qpe_circuit({m}) uses {', '.join(big)} on 3 or more qubits. Use only one- and two-qubit gates "
                           "(cp, swap, h, x): the course noise model puts errors only on those."]
        mm = measurement_map(qc)
        if mm != {k: k for k in range(m)}:
            hint = ""
            if m in mm:
                hint = " Do not measure the target qubit: it stays in |1>."
            return False, [f"qpe_circuit({m}) should measure counting qubit k into classical bit k, for k = 0 to {m - 1} "
                           f"(found {mm or 'no measurements'}).{hint}"]
        try:
            got = _counting_probs(qc, m)
        except Exception as e:  # noqa: BLE001
            return False, [f"The circuit from qpe_circuit({m}) cannot be simulated: {type(e).__name__}: {e}"]
        want = _counting_probs(reference_qpe(m), m)
        if np.abs(got - want).max() > 1e-9:
            y = int(np.argmax(got))
            hint = ""
            if np.abs(got - _counting_probs(reference_qpe(m, 1 - PHASE), m)).max() < 1e-9:
                hint = (" The circuit estimates 2/3 instead of 1/3: the controlled phases have the wrong sign, or the "
                        "QFT is not inverted (use the prepared inverse_qft).")
            elif got[0] > 0.999:
                hint = " The counting qubits end in |0...0>: check the X on the target qubit and the cp angles."
            elif abs(got[int(format(best_estimate(m), f'0{m}b')[::-1], 2)] - want[best_estimate(m)]) < 1e-9:
                hint = " The bits come out in reverse order: counting qubit k should control cp(2*pi*phase*2**k)."
            return False, [f"qpe_circuit({m}) gives a different distribution from the expected one (most likely y = {y}, "
                           f"expected y = {best_estimate(m)}).{hint}"]
        n2, want2 = two_qubit_count(qc), ops_two_qubit_count(reference_qpe_ops(m))
        if n2 != want2:
            return False, [f"qpe_circuit({m}) gives the right distribution but has {n2} two-qubit gates; the project's "
                           f"circuit has {want2}. Use one cp(2 * pi * (1/3) * 2**k, k, m) for each counting qubit k (not "
                           "2**k repeated gates) and the prepared inverse_qft: with noise, every extra gate changes the result."]
    return True, ["qpe_circuit(): passed (2 to 5 counting qubits)."]


ERROR_CASES = [({"011": 1000}, 3), ({"010": 500, "011": 500}, 3), ({"0101": 3, "1111": 1}, 4),
               ({"000": 1, "100": 1, "111": 2}, 3), ({"01011": 2500, "01010": 1000, "11011": 500}, 5)]


def _test_error(estimate_error):
    for counts, m in ERROR_CASES:
        total = sum(counts.values())
        want = sum(v * circular_distance(int(k, 2), m) for k, v in counts.items()) / total
        try:
            got = estimate_error(dict(counts), m)
        except NotImplementedError:
            return False, ["estimate_error() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"estimate_error({counts}, {m}) raised {type(e).__name__}: {e}"]
        try:
            gotf = float(got)
        except (TypeError, ValueError):
            return False, [f"estimate_error({counts}, {m}) should return a number; it returned {got!r}."]
        if abs(gotf - want) > 1e-9:
            plain = sum(v * abs(int(k, 2) / 2 ** m - PHASE) for k, v in counts.items()) / total
            rev = sum(v * circular_distance(int(k[::-1], 2), m) for k, v in counts.items()) / total
            unweighted = np.mean([circular_distance(int(k, 2), m) for k in counts])
            hint = ""
            if abs(gotf - plain) < 1e-9:
                hint = " Measure the distance around the circle: min(d, 1 - d), so that 0.875 is 0.458 away from 1/3, not 0.542."
            elif abs(gotf - rev) < 1e-9:
                hint = " Read each key with int(key, 2): the leftmost character is the highest bit."
            elif abs(gotf - unweighted) < 1e-9:
                hint = " Weight each result by its count, and divide by the total number of shots."
            return False, [f"estimate_error({counts}, {m}) returned {gotf:.6f}; expected {want:.6f}.{hint}"]
    return True, [f"estimate_error(): passed ({len(ERROR_CASES)} cases)."]


def _test_counts(my_counts, m, p):
    if not isinstance(my_counts, dict) or not my_counts:
        return False, ["my_counts is empty. Run the cell of Step 8 that makes it first."]
    total = counts_total(my_counts)
    if total != SHOTS:
        return False, [f"my_counts holds {total} shots; use shots=4000 (Step 8)."]
    if any(len(str(k).replace(" ", "")) != m for k in my_counts):
        return False, [f"The keys of my_counts should be {m}-bit strings: run qpe_circuit(COUNT) with COUNT = {m}."]
    best = format(best_estimate(m), f"0{m}b")
    k = int(my_counts.get(best, 0))
    sigma = math.sqrt(p * (1 - p) / SHOTS)
    z = (k / SHOTS - p) / sigma
    if abs(z) > 3:
        return False, [f"In my_counts, {k} of 4000 shots gave the best estimate {best}: a fraction {k / SHOTS:.4f}, "
                       f"{z:+.1f} standard deviations from the noise model's expected value {p:.4f}. Check that my_counts "
                       "comes from qpe_circuit(COUNT) run with course_noise_model(P2, READOUT) and 4,000 shots. If it does, "
                       "a result this far out happens by chance about 3 times in 1,000: run Step 8 again with another seed."]
    return True, [f"my_counts: {k} of 4000 shots gave the best estimate {best} ({k / SHOTS:.4f}), within 3 standard "
                  f"deviations ({z:+.2f}) of the expected {p:.4f}."]


def personal_value(count, p2, readout):
    return int(round(10000 * expected_best(int(count), float(p2), float(readout))))


def check_l2_m6_qpe(qpe_circuit, estimate_error, shot_sigma, within_3_sigma, difference_sigma, COUNT, P2, READOUT, my_counts):
    try:
        m = int(COUNT)
        if m != float(COUNT) or not 3 <= m <= 6:
            raise ValueError
    except (TypeError, ValueError):
        return False, ["COUNT is a whole number from 3 to 6. Copy it again from your project quiz."], None
    ok, msg, p2, ro = personal_noise_ok(P2, READOUT)
    if not ok:
        return False, [msg], None
    msgs, passed = [], True
    for good, mm in (test_statistics(shot_sigma, within_3_sigma, difference_sigma), _test_qpe(qpe_circuit),
                     _test_error(estimate_error)):
        msgs += mm
        passed = passed and good
        if not good:
            break
    if not passed:
        return False, msgs, None
    p = expected_best(m, p2, ro)
    good, mm = _test_counts(my_counts, m, p)
    msgs += mm
    if not good:
        return False, msgs, None
    return True, msgs, int(round(10000 * p))


passed, messages, value = check_l2_m6_qpe(qpe_circuit, estimate_error, shot_sigma, within_3_sigma, difference_sigma,
                                          COUNT, P2, READOUT, my_counts)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **A model, not a computer.** The course model has the same depolarizing error on every gate and qubit. A real computer has a different error on each qubit pair, drifts between calibrations, and has errors that repeat the same way every time (a phase that is slightly off), which add up instead of averaging out.
- **No connectivity.** The inverse QFT has a `cp` between every pair of counting qubits. On a real chip most pairs are not connected, so the transpiler adds swaps (Module 4), and the best m would be smaller still.
- **One phase.** 1/3 is a hard case: it is never an m-bit fraction. A phase such as 1/4 has an exact 2-bit answer, and its ideal probability is 1.
- **Shot noise.** Each mean error is uncertain by its SEM; the data cannot rank two values of m whose difference is below 3σ_diff, even when the model can.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level2/QC-L2-M6-hardware-colab.ipynb) runs the project's reference circuit on a real computer and compares the result with the ideal value, with this course's noise model, and with IBM's own noise model of that computer. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days, and one run of 4,000 shots uses a few seconds of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Optional: a short summary of your experiment

Not graded. Writing five short answers is a good way to prepare for the project quiz, and it is how results are reported in a lab notebook or a paper. Double-click this cell to edit it.

1. **Question.** What did you want to find out?
2. **Method.** Which circuit, which noise model (P2, READOUT), how many shots?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.687 ± 0.007, with σ from Step 1).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the Module 6 project quiz for this experiment, phase estimation with noise, and the Module 6 time log.